## Forest Cover Type Prediction

## Project Objective

The objective of this project is to predict seven different forest cover types
in four different wilderness areas of the Roosevelt National Forest of
Northern Colorado.

This project includes:
1. Data loading
2. Data understanding
3. Exploratory Data Analysis
4. Data preprocessing
5. Feature engineering
6. Model building
7. Model evaluation
8. Hyperparameter tuning
9. Model comparison
10. Challenges and solutions
11. Final conclusion

In [ ]:
### important libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
accuracy_score,
precision_score,
recall_score,
f1_score,
classification_report,
confusion_matrix
)
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv(r"train.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.columns

In [ ]:
df.shape

## Checking Missing Values

In [ ]:
df.isnull().sum().sum()

In [ ]:
df.nunique()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates().reset_index(drop=True)
df.duplicated().sum()

In [ ]:
df.head()

## Check Traget Variables

In [ ]:
df["Cover_Type"].value_counts().sort_index()

In [ ]:
print("Number of cover types:",df["Cover_Type"].nunique())

In [ ]:
df["Cover_Type"].unique()

## Visualize Target Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(
    data=df,
    x="Cover_Type"
)

plt.title("Dribution of Forest Cover Types")
plt.xlabel("Cover Type")
plt.ylabel("Number of samples")

plt.show()

In [ ]:
numerical_columns = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]

In [ ]:
df[numerical_columns].describe()

In [ ]:
df[numerical_columns].hist(
    figsize=(16,12),
    bins=30
)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,8))
df[numerical_columns].boxplot()
plt.xticks(rotation=45)
plt.title("Boxplot of Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
correlation = df[numerical_columns].corr()
plt.figure(figsize=(12,8))
sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt="2f"
)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
wilderness_columns = [
    col for col in df.columns
    if col.startswith("Wilderness_Area")
]
wilderness_columns

In [ ]:
df[wilderness_columns].sum()

In [ ]:
df[wilderness_columns].sum().plot(
    kind="bar",
    figsize=(8,5)
)
plt.title("Sample in each wilderness Area")
plt.xlabel("Wilderness Area")
plt.ylabel("Number of Samples")
plt.show()

In [ ]:
soil_columns = [
    c for c in df.columns
    if c.startswith("Soil_type")
]
print("Number of soil column:",len(soil_columns))

In [ ]:
df[soil_columns].sum().sort_values(ascending=False).head(10)

## Train Test split

In [ ]:
x = df.drop("Cover_Type",axis=1)
y = df["Cover_Type"]

In [ ]:
x.shape

In [ ]:
y.shape

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## Feature Scaling

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

## Models

In [ ]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
logistic_model.fit(x_train_scaled,y_train)
logistic_pred = logistic_model.predict(x_test_scaled)

In [ ]:
print(
    classification_report(
        y_test,
        logistic_pred
    )
)

In [ ]:
decision_tree = DecisionTreeClassifier(
    random_state=42
)
decision_tree.fit(
    x_train,
    y_train
)
dt_pred = decision_tree.predict(
    x_test
)

In [ ]:
print(classification_report(y_test,dt_pred))

In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs =-1
)
random_forest.fit(
    x_train,
    y_train
)
rf_pred = random_forest.predict(x_test)

In [ ]:
print(classification_report(y_test,rf_pred))

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train_scaled,y_train)
knn_pred = knn_model.predict(x_test_scaled)

In [ ]:
print(classification_report(y_test,knn_pred))

In [ ]:
svm_model = SVC(kernel="rbf")
svm_model.fit(x_train_scaled,y_train)
svm_pred = svm_model.predict(x_test_scaled)

In [ ]:
print(classification_report(y_test,svm_pred))

In [ ]:
print(classification_report(y_test,svm_pred))

In [ ]:
gradient_model = GradientBoostingClassifier(random_state=42)
gradient_model.fit(x_train,y_train)
gradient_pred = gradient_model.predict(x_test)

In [ ]:
print(classification_report(y_test,gradient_pred))

## Model Comparison

In [ ]:
model = {
    "Logistic Regression": (y_test,logistic_pred),
    "Decision Tree": (y_test,dt_pred),
    "Random Forest": (y_test,rf_pred),
    "KNN": (y_test,knn_pred),
    "SVM": (y_test,svm_pred),
    "Gradient Boosting": (y_test,gradient_pred)
}

In [ ]:
result = []
for model_name,(actual,predicted) in model.items():
    accuracy = accuracy_score(
        actual,predicted
    )
    precision = precision_score(
        actual,predicted,average="weighted"
    )
    recall = recall_score(
        actual,predicted,average="weighted"
    )
    f1 = f1_score(
        actual,predicted,average="weighted"
    )
    result.append({
        "Model":model_name,
        "Accuracy":accuracy,
        "Precision":precision,
        "Recall":recall,
        "F1 Score":f1
    })

result_df = pd.DataFrame(result)
result_df

In [ ]:
result_df.set_index("Model")[
    ["Accuracy","Precision","Recall","F1 Score"]
].plot(
    kind="bar",
    figsize=(12,6)
)
plt.title("Model performance Comparision")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(
    y_test,rf_pred
)
plt.figure(figsize=(8,6))
sns.heatmap(
    cm,annot=True,fmt="d",cmap="Blues"
)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    "Feature":x.columns,
    "Importance":random_forest.feature_importances_
})
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)
feature_importance.head(20)

In [ ]:
plt.figure(figsize=(10,8))
sns.barplot(
    data=feature_importance.head(20),
    x="Importance",
    y="Feature"
)
plt.title("Top 20 Important Features")
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    "n_estimators":[100,200],
    "max_depth":[None,20,30],
    "min_samples_split":[2,5],
    "min_samples_leaf":[1,2]
}

In [ ]:
grid_search = GridSearchCV(
    RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ),
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1,
    verbose=1
)

In [ ]:
grid_search.fit(x_train,y_train)

In [ ]:
print("Best Parameters:")
print(grid_search.best_params_)

In [ ]:
print("Best Cross Validation Score:")
print(grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_
best_pred = best_model.predict(x_test)

In [ ]:
print(classification_report(y_test,best_pred))

## Final Model Metrics

In [ ]:
final_accuracy = accuracy_score(
    y_test,
    best_pred
)
final_precision = precision_score(
    y_test,
    best_pred,
    average="weighted"
)
final_recall = recall_score(
    y_test,
    best_pred,
    average="weighted"
)
final_f1 = f1_score(
    y_test,
    best_pred,
    average="weighted"
)
print("Accuracy:",final_accuracy)
print("Precision:",final_precision)
print("Recall:",final_recall)
print("F1:",final_f1)

In [ ]:
cm = confusion_matrix(
    y_test,
    best_pred
)
plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)
plt.title("Final MOdel Confusion Matrix")
plt.xlabel("Predicted Cover Type")
plt.ylabel("Actual Cover Type")
plt.show()

In [ ]:
print(classification_report(y_test,best_pred))

## Challenges
Handling a dataset with many features.

Scaling features for some machine learning models.

Classifying seven different forest cover types.

Comparing multiple machine learning algorithms.

Finding suitable hyperparameters for better performance.

## Limitations
Dataset is limited to a specific geographical region.

Other environmental factors such as weather are not included.

Model accuracy is not 100%.

Performance depends on the quality of the dataset.

## Conclusion

The project successfully predicted forest cover types using machine learning. Different classification algorithms were compared, and Random Forest achieved the best performance with about 85.98% accuracy. The project demonstrates that machine learning can be effectively used for forest cover classification.